In [3]:
from ipywidgets import widgets
from IPython.display import display

uploader = widgets.FileUpload(accept=".png", multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [8]:
print(uploader.value)

({'name': 'Screenshot from 2026-07-05 13-28-31.png', 'type': 'image/png', 'size': 198734, 'content': <memory at 0x71fa89183b80>, 'last_modified': datetime.datetime(2026, 7, 5, 7, 58, 31, 822000, tzinfo=datetime.timezone.utc)},)


In [10]:
import base64

try:
  uploaded_file = uploader.value[0]
  content_mv = uploaded_file["content"]
  img_bytes = bytes(content_mv)

  img_b64 = base64.b64encode(img_bytes).decode("utf-8")
except Exception as e:
  print("error while image encoding: ",e)

In [1]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from tavily.client import TavilyClient
from langchain.tools import tool
from typing import Dict, Any
tavily_client = TavilyClient()

@tool(description="search web for food recipe")
def web_search(q:str) -> Dict[str, Any]:
  return tavily_client.search(query=q)

model = init_chat_model(
  model="qwen/qwen3.6-27b",
  model_provider="groq",
  temperature=0.7
)

agent = create_agent(
  model=model,
  tools=[web_search],
  checkpointer=InMemorySaver(),
  system_prompt="""you are a Master Chef , help user with food related queries,
  and just be precies if users asks for some recipe then give them a recipe by searching the web,
  dont list the ingredients until specifically asked to

  give short sweet answer no big paragraphs, unless asked to
  """,
)

config = {"configurable": {"thread_id":"chat_02"}}

while(True):
  userInput = input("Ask ChefAgent anything: ")
  if userInput.lower() == "exit":
    break
  multimodal_query = HumanMessage(content=[
    {"type":"text", "text":userInput},
    {"type":"image", "base64":img_b64, "mime_type":"image/png"}
  ])

  response = agent.invoke(
    {"messages": [multimodal_query]},
    config
  )
  print(response["messages"][-1].content)
